# 01 — Run ExperimentRuns LLM API calls and saves results to Google Drive.**First time?** See README.md for setup instructions.

## 1. Clone Repo & Mount Drive

In [ ]:
# Clone your repo (only needs to run once per Colab session)!git clone https://github.com/auertobias/authority-bias-llm.git 2>/dev/null || echo "Already cloned"%cd authority-bias-llm# Mount Google Drivefrom google.colab import drivedrive.mount('/content/drive')

In [ ]:
# Install dependencies!pip install -q google-genai openai anthropic

## 2. Configure

In [ ]:
import os, syssys.path.insert(0, '.')from src.config import DATA_PATH, RESULTS_PATH, N_REPS, PAUSE_SECONDS, CHECKPOINT_EVERYos.makedirs(DATA_PATH, exist_ok=True)os.makedirs(RESULTS_PATH, exist_ok=True)print(f"Data saves to: {DATA_PATH}")

In [ ]:
# ── API Keys ───────────────────────────────────────────────# Use Colab Secrets (🔑 icon in sidebar) — add your keys there.from google.colab import userdata# Uncomment the keys you have:# GEMINI_API_KEY    = userdata.get('GEMINI_API_KEY')# OPENAI_API_KEY    = userdata.get('OPENAI_API_KEY')# ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

## 3. Load Data & Build Trials

In [ ]:
import pandas as pdarguments   = pd.read_csv(DATA_PATH + "arguments.csv")authorities = pd.read_csv(DATA_PATH + "authority.csv")print(f"Arguments:   {len(arguments)} rows, {arguments['claim_id'].nunique()} claims")print(f"Authorities: {len(authorities)} rows")

In [ ]:
trials = []trial_id = 0for _, arg in arguments.iterrows():    for _, auth in authorities.iterrows():        if auth['branch'] == 'none':            expertise = 'irrelevant'        elif auth['branch'] == arg['branch']:            expertise = 'relevant'        else:            expertise = 'irrelevant'        trial_id += 1        trials.append({            'trial_id': trial_id,            'argument_id': arg['id'],            'claim': arg['claim'],            'argument': arg['argument'],            'stance': arg['stance'],            'arg_type': arg['type'],            'arg_branch': arg['branch'],            'authority_id': auth['authority_id'],            'authority_label': auth['label'].strip(),            'status': auth['status'],            'auth_branch': auth['branch'],            'expertise': expertise,        })trials_df = pd.DataFrame(trials)trials_df = trials_df.sample(frac=1, random_state=42).reset_index(drop=True)print(f"Total trials: {len(trials_df)}")print(f"\nTrials per cell:")print(trials_df.groupby(['status', 'expertise', 'arg_type']).size().unstack(fill_value=0))

## 4. Choose Models & Run

In [ ]:
from src.models import make_gemini_fn, make_gpt_fn, make_claude_fnfrom src.prompts import build_promptfrom src.parsing import extract_rating, extract_weaknesses# Uncomment the models you want to run:MODELS = {    # 'gemini': make_gemini_fn(GEMINI_API_KEY),    # 'gpt':    make_gpt_fn(OPENAI_API_KEY),    # 'claude': make_claude_fn(ANTHROPIC_API_KEY),}if not MODELS:    print("⚠️  No models selected! Uncomment at least one line above.")

In [ ]:
import timefrom datetime import datetimefor model_name, model_fn in MODELS.items():    print(f"\n{'='*60}")    print(f"  RUNNING: {model_name.upper()}")    print(f"{'='*60}")    results = []    for rep in range(N_REPS):        run_order = trials_df.sample(frac=1, random_state=rep*1000).reset_index(drop=True)        print(f"\n--- Rep {rep+1}/{N_REPS} ({len(run_order)} trials) ---")        for idx, row in run_order.iterrows():            prompt = build_prompt(row)            raw = model_fn(prompt)            rating = extract_rating(raw)            weaknesses = extract_weaknesses(raw)            results.append({                'trial_id': row['trial_id'],                'rep': rep + 1,                'model': model_name,                'status': row['status'],                'expertise': row['expertise'],                'arg_type': row['arg_type'],                'arg_branch': row['arg_branch'],                'authority_label': row['authority_label'],                'stance': row['stance'],                'claim': row['claim'],                'argument': row['argument'],                'rating': rating,                'weaknesses': weaknesses,                'raw_response': raw,            })            n_done = idx + 1            if n_done % 50 == 0:                print(f"  {n_done}/{len(run_order)} done | last rating: {rating}")            if n_done % CHECKPOINT_EVERY == 0:                pd.DataFrame(results).to_csv(                    DATA_PATH + f"checkpoint_{model_name}.csv", index=False                )            time.sleep(PAUSE_SECONDS)    # Save final    date_str = datetime.now().strftime("%Y%m%d")    filename = f"data_{model_name}_{date_str}.csv"    filepath = DATA_PATH + filename    pd.DataFrame(results).to_csv(filepath, index=False)    valid = sum(1 for r in results if r['rating'] is not None)    print(f"\n✓ Saved: {filename}")    print(f"  {len(results)} rows, {valid} valid ratings ({100*valid/len(results):.0f}%)")

## 5. Verify

In [ ]:
import globdata_files = sorted(glob.glob(DATA_PATH + "data_*.csv"))print("Data files on Drive:")for f in data_files:    df_check = pd.read_csv(f)    valid = df_check['rating'].notna().sum()    print(f"  {os.path.basename(f):40s} → {len(df_check)} rows, {valid} valid")